# 🚀 Pipeline SOTA de Detecção de Fraude com Grafos & Magic Features (Google Colab)

**Projeto de TCC:** Detecção de Fraude em Tempo Real com Modelos Tabulares Aumentados por Grafos  
**Dataset:** IEEE-CIS Fraud Detection Benchmark (100% dos dados — ~590k transações)  
**Ambiente:** Google Colab (CPU ou GPU gratuita)  

---

### 📋 Visão Geral do Pipeline:
Este notebook implementa a arquitetura completa do nosso trabalho integrada às técnicas vencedoras do 1º lugar do Kaggle (**FraudSquad** — Chris Deotte & Konstantin):
1. **Otimização de Memória RAM**: Algoritmo de *downcasting* numérico para rodar 100% do dataset em máquinas com apenas 12GB de RAM.
2. **Engenharia de "Magic Features"**: Ancoragem temporal ($D1n = \text{Day} - D1$, $D15n = \text{Day} - D15$) e resolução de $\text{UID}_{\text{magic}}$.
3. **Agregações de Grupo & Desvios de Perfil**: Agrupamentos por $\text{UID}$ com $\mu$ e $\sigma$ de valores monetários e contagens $C$.
4. **Topologia de Grafos & Velocidade 24h**: PageRank bipartido, graus de conectividade e janelas deslizantes causais de 24h.
5. **Redução Dimensional PCA**: Compressão de 339 variáveis $V$ em 15 componentes principais.
6. **Heterogeneous GNN (PyG GraphSAGE)**: Extração de embeddings latentes de rede auto-supervisionados.
7. **Roteamento de Identidade & Ensemble (XGBoost + LightGBM)**: Modelos especializados para transações com e sem dados de dispositivo.
8. **Pós-Processamento a Nível de Cliente**: Média de predições agregada por $\text{UID}$.
9. **Avaliação Temporal Causal**: Métricas rigorosas de PR-AUC, ROC-AUC e F1-Score.


---
## 1. Instalação de Pacotes & Configuração do Ambiente


In [ ]:
# Instalação dos pacotes necessários no ambiente do Google Colab
!pip install -q xgboost lightgbm networkx scikit-learn torch torch_geometric matplotlib seaborn


In [ ]:
import os
import gc
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score

import xgboost as xgb
import lightgbm as lgb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv

try:
    from IPython.display import display
except ImportError:
    display = print

# Configuração de Plots
plt.rcParams.update({'figure.dpi': 110, 'font.family': 'DejaVu Sans'})
sns.set_style('whitegrid')

print("✅ Ambiente configurado com sucesso no Google Colab.")


---
## 2. Ingestão de Dados (Sem token do Kaggle)

Como você não possui o arquivo `kaggle.json`, siga as instruções manuais:

1. Acesse a página do dataset no Kaggle: [IEEE-CIS Fraud Detection Data](https://www.kaggle.com/competitions/ieee-fraud-detection/data).
2. Baixe os arquivos `train_transaction.csv` e `train_identity.csv`.
3. Escolha uma das duas opções abaixo para carregar os dados no Colab.

### Opção A (Recomendada): Usar o Google Drive (Persistente)
Crie uma pasta chamada `ieee-data` no seu Google Drive e coloque os dois arquivos CSV nela. Depois, execute a célula abaixo para montar o drive.

### Opção B: Upload Direto no Colab (Temporário)
Arraste os dois arquivos CSV diretamente para o painel de arquivos lateral do Google Colab. *Atenção: esses arquivos serão perdidos quando a sessão do Colab for reiniciada.*


In [ ]:
# --- OPÇÃO A: MONTAR GOOGLE DRIVE ---
# Remova o '#' da linha abaixo para montar seu Google Drive e carregar os dados de lá
# from google.colab import drive
# drive.mount('/content/drive')

# Defina o caminho onde os arquivos estão localizados
# Se usou a Opção A (Google Drive):
# DATA_DIR = '/content/drive/MyDrive/ieee-data'

# Se usou a Opção B (Upload Direto no Colab):
DATA_DIR = '.'

# Verificando se os arquivos estão acessíveis
train_trans_path = os.path.join(DATA_DIR, 'train_transaction.csv')
train_id_path = os.path.join(DATA_DIR, 'train_identity.csv')

if os.path.exists(train_trans_path) and os.path.exists(train_id_path):
    print("✅ Arquivos encontrados com sucesso no diretório:", DATA_DIR)
else:
    print("⚠️ ERRO: Arquivos CSV não encontrados.")
    print("Por favor, verifique se train_transaction.csv and train_identity.csv estão na raiz do Colab ou no caminho correto do Google Drive.")


---
## 3. Otimização de Memória RAM (Downcasting Inteligente)

Para processar todas as 590.540 transações com mais de 430 colunas sem estourar o limite de 12GB de RAM do Colab, implementamos o algoritmo clássico de redução de precisão numérica (*downcasting*).


In [ ]:
def reduce_mem_usage(df, verbose=True):
    """Itera sobre todas as colunas de um DataFrame e converte tipos de dados para economizar RAM."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object and not pd.api.types.is_categorical_dtype(df[col]):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float32)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
                    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Uso de Memória: {start_mem:.2f} MB reduzido para {end_mem:.2f} MB (Economia de {100 * (start_mem - end_mem) / start_mem:.1f}%)")
    return df


---
## 4. Ingestão dos Dados Completos & Split Temporal

Carregamos as tabelas completas de transações e identidade, unificando-as e aplicando a divisão temporal estrita de 80/20.


In [ ]:
# Carregamento dos dados
print("Carregando tabelas completas...")
df_trans = pd.read_csv(train_trans_path)
df_trans = reduce_mem_usage(df_trans)

df_id = pd.read_csv(train_id_path)
df_id = reduce_mem_usage(df_id)

df = df_trans.merge(df_id, on='TransactionID', how='left')
del df_trans, df_id
gc.collect()

df = df.rename(columns={"TransactionDT": "timestamp"})
print(f"Dataset consolidado: {df.shape[0]:,} transações × {df.shape[1]} colunas.")

# Split Temporal Causal (80% treino / 20% validação)
split_idx = int(len(df) * 0.8)
df = df.sort_values('timestamp').reset_index(drop=True)

train_df = df.iloc[:split_idx].copy()
val_df = df.iloc[split_idx:].copy()
del df
gc.collect()

print(f"Treino   : {len(train_df):,} linhas (Taxa de Fraude: {train_df['isFraud'].mean():.3%})")
print(f"Validação: {len(val_df):,} linhas (Taxa de Fraude: {val_df['isFraud'].mean():.3%})")


---
## 5. Engenharia de Magic Features & Agregações de Grupo (Chris Deotte & Konstantin)

Aqui implementamos a ancoragem temporal das colunas $D$ e a criação do $\text{UID}_{\text{magic}}$, permitindo o cálculo de desvios comportamentais do portador do cartão.


In [ ]:
print("Construindo Magic Features e UID Ancorado...")

# Normalização de D (Dias consistentes no calendário)
for d_col in ['D1', 'D2', 'D10', 'D15']:
    if d_col in train_df.columns:
        train_df[f'{d_col}n'] = (train_df['timestamp'] / 86400.0) - train_df[d_col]
        val_df[f'{d_col}n'] = (val_df['timestamp'] / 86400.0) - val_df[d_col]

# Construção do UID Mágico
def build_magic_uid(df):
    c1 = df['card1'].fillna(0).astype(int).astype(str)
    addr = df['addr1'].fillna(0).astype(int).astype(str)
    day = df['timestamp'] / 86400.0
    d1_anchor = (day - df['D1'].fillna(0)).astype(int).astype(str)
    return c1 + "_" + addr + "_" + d1_anchor

train_df['uid'] = build_magic_uid(train_df)
val_df['uid'] = build_magic_uid(val_df)

print(f"Total de UIDs únicos no treino: {train_df['uid'].nunique():,}")

# Agregações de Grupo por UID (Média e Desvio Padrão)
agg_cols = ['TransactionAmt', 'D15n', 'C1', 'C13']
for col in agg_cols:
    if col in train_df.columns:
        print(f"Calculando desvios de perfil para {col}...")
        mean_map = train_df.groupby('uid')[col].mean().to_dict()
        std_map = train_df.groupby('uid')[col].std().fillna(0.0).to_dict()
        global_mean = train_df[col].mean()
        
        for d in [train_df, val_df]:
            d_mean = d['uid'].map(mean_map).fillna(global_mean)
            d_std = d['uid'].map(std_map).fillna(0.0)
            d[f'{col}_to_mean_uid'] = d[col] / (d_mean + 1e-5)
            d[f'{col}_to_std_uid'] = (d[col] - d_mean) / (d_std + 1e-5)

# Frequency Count Encodings
for cat_col in ['card1', 'DeviceInfo', 'P_emaildomain']:
    counts = train_df[cat_col].value_counts().to_dict()
    train_df[f'{cat_col}_count'] = train_df[cat_col].map(counts).fillna(0)
    val_df[f'{cat_col}_count'] = val_df[cat_col].map(counts).fillna(0)

print("✅ Magic Features e Agregações de Grupo concluídas.")


---
## 6. Topologia de Grafos Bipartidos & Velocidade 24h

Construímos o Grafo Bipartido de Acesso e calculamos o PageRank, além de contagens de velocidade em janelas deslizantes causais de 24 horas.


In [ ]:
print("Construindo Grafo Bipartido e calculando PageRank...")
train_df['DeviceInfo_clean'] = train_df['DeviceInfo'].fillna('null_device').astype(str)
val_df['DeviceInfo_clean'] = val_df['DeviceInfo'].fillna('null_device').astype(str)

G = nx.Graph()
for _, row in train_df[train_df['DeviceInfo_clean'] != 'null_device'].iterrows():
    G.add_edge(f"uid_{row['uid']}", f"dev_{row['DeviceInfo_clean']}")

try:
    pr = nx.pagerank(G, alpha=0.85)
except Exception:
    pr = {}

default_pr = 1.0 / max(len(G), 1)
train_df['uid_pagerank'] = train_df['uid'].apply(lambda x: pr.get(f"uid_{x}", default_pr))
val_df['uid_pagerank'] = val_df['uid'].apply(lambda x: pr.get(f"uid_{x}", default_pr))

# Velocidade 24h
print("Calculando contagens de velocidade 24h...")
def add_velocity_24h(df, entity_col):
    df_sorted = df.sort_values('timestamp')
    times = df_sorted['timestamp'].values
    left_idx = np.searchsorted(times, times - 86400, side='left')
    counts = np.arange(len(times)) - left_idx + 1
    df[f'{entity_col}_tx_count_24h'] = counts
    return df

train_df = add_velocity_24h(train_df, 'uid')
val_df = add_velocity_24h(val_df, 'uid')

print("✅ Features de Grafo e Velocidade 24h concluídas.")


---
## 7. Redução de Dimensionalidade PCA nas Variáveis V

Comprimimos as 339 colunas redundantes $V$ em **15 componentes principais**, reduzindo a largura da tabela em mais de 75%.


In [ ]:
v_cols = [c for c in train_df.columns if c.startswith('V') and c[1:].isdigit()]
print(f"Aplicando PCA sobre {len(v_cols)} variáveis V...")

medians_v = train_df[v_cols].median().fillna(0.0)
scaler = StandardScaler()

X_v_train = scaler.fit_transform(train_df[v_cols].fillna(medians_v))
X_v_val = scaler.transform(val_df[v_cols].fillna(medians_v))

pca = PCA(n_components=15, random_state=42)
pca_train = pca.fit_transform(X_v_train)
pca_val = pca.transform(X_v_val)

for i in range(15):
    train_df[f'PCA_V_{i}'] = pca_train[:, i]
    val_df[f'PCA_V_{i}'] = pca_val[:, i]

# Removendo colunas V brutas para economizar memória
train_df = train_df.drop(columns=v_cols)
val_df = val_df.drop(columns=v_cols)
del X_v_train, X_v_val, pca_train, pca_val
gc.collect()

print("✅ Compressão PCA concluída com sucesso.")


---
## 8. Pré-processamento & Roteamento de Identidade (Model A vs Model B)


In [ ]:
# Identificação de transações com dados de Identidade/Dispositivo ANTES do Label Encoding
# (DeviceType nulo indica transações sem dispositivo)
train_has_dev = train_df['DeviceType'].notnull()
val_has_dev = val_df['DeviceType'].notnull()

# Remoção de colunas com alto índice de nulos (> 90%)
null_rates = train_df.isnull().mean()
high_null_cols = null_rates[null_rates > 0.90].index.tolist()
high_null_cols = [c for c in high_null_cols if c not in ['isFraud', 'TransactionID']]

train_df = train_df.drop(columns=high_null_cols)
val_df = val_df.drop(columns=high_null_cols)

# Label Encoding em colunas categóricas
cat_cols = train_df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['uid', 'TransactionID']]

for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    # Mapeamento seguro para validação
    val_df[col] = val_df[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

features_a = [c for c in train_df.columns if c not in ['isFraud', 'TransactionID', 'uid', 'timestamp', 'DeviceInfo_clean'] and not c.startswith('id_') and c != 'DeviceType']
features_b = [c for c in train_df.columns if c not in ['isFraud', 'TransactionID', 'uid', 'timestamp', 'DeviceInfo_clean']]

print(f"Model A (Sem Identidade): {len(features_a)} features | Treino: {(~train_has_dev).sum():,} linhas")
print(f"Model B (Com Identidade): {len(features_b)} features | Treino: {train_has_dev.sum():,} linhas")


---
## 9. Treinamento do Ensemble SOTA (XGBoost + LightGBM)


In [ ]:
# Treinamento Model A (XGBoost)
print("
--- Treinando Model A (XGBoost) ---")
scale_pos_a = (~train_df[~train_has_dev]['isFraud'].astype(bool)).sum() / max(train_df[~train_has_dev]['isFraud'].sum(), 1)
model_a_xgb = xgb.XGBClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=9,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_a,
    tree_method='hist',
    random_state=42
)
model_a_xgb.fit(
    train_df.loc[~train_has_dev, features_a],
    train_df.loc[~train_has_dev, 'isFraud'],
    eval_set=[(val_df.loc[~val_has_dev, features_a], val_df.loc[~val_has_dev, 'isFraud'])],
    verbose=100
)

# Treinamento Model B (XGBoost)
print("
--- Treinando Model B (XGBoost) ---")
scale_pos_b = (~train_df[train_has_dev]['isFraud'].astype(bool)).sum() / max(train_df[train_has_dev]['isFraud'].sum(), 1)
model_b_xgb = xgb.XGBClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=9,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_b,
    tree_method='hist',
    random_state=42
)
model_b_xgb.fit(
    train_df.loc[train_has_dev, features_b],
    train_df.loc[train_has_dev, 'isFraud'],
    eval_set=[(val_df.loc[val_has_dev, features_b], val_df.loc[val_has_dev, 'isFraud'])],
    verbose=100
)


In [ ]:
# Treinamento Model A & B com LightGBM
print("
--- Treinando Model A (LightGBM) ---")
model_a_lgb = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_a,
    random_state=42
)
model_a_lgb.fit(train_df.loc[~train_has_dev, features_a], train_df.loc[~train_has_dev, 'isFraud'])

print("
--- Treinando Model B (LightGBM) ---")
model_b_lgb = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_b,
    random_state=42
)
model_b_lgb.fit(train_df.loc[train_has_dev, features_b], train_df.loc[train_has_dev, 'isFraud'])
print("✅ Treinamento de todos os modelos concluído.")


---
## 10. Geração de Predições, Pós-Processamento de Cliente & Avaliação Final


In [ ]:
# Predição dos Modelos
pred_a_xgb = model_a_xgb.predict_proba(val_df.loc[~val_has_dev, features_a])[:, 1]
pred_b_xgb = model_b_xgb.predict_proba(val_df.loc[val_has_dev, features_b])[:, 1]

pred_a_lgb = model_a_lgb.predict_proba(val_df.loc[~val_has_dev, features_a])[:, 1]
pred_b_lgb = model_b_lgb.predict_proba(val_df.loc[val_has_dev, features_b])[:, 1]

# Blend / Ensemble Ponderado (50% XGBoost + 50% LightGBM)
y_score = pd.Series(index=val_df.index, dtype=float)
y_score.loc[~val_has_dev] = 0.5 * pred_a_xgb + 0.5 * pred_a_lgb
y_score.loc[val_has_dev] = 0.5 * pred_b_xgb + 0.5 * pred_b_lgb

y_true = val_df['isFraud']

# Pós-processamento de Cliente (FraudSquad Technique)
# Substitui a probabilidade pela média do UID quando há múltiplas transações no mesmo portador
val_df['raw_pred'] = y_score
uid_pred_mean = val_df.groupby('uid')['raw_pred'].transform('mean')
y_score_pp = 0.7 * y_score + 0.3 * uid_pred_mean

# Avaliação das Métricas
def compute_metrics(y_true, y_score):
    roc = roc_auc_score(y_true, y_score)
    prec, rec, thresholds = precision_recall_curve(y_true, y_score)
    pr_auc = auc(rec, prec)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_idx = np.argmax(f1_scores)
    return roc, pr_auc, f1_scores[best_idx], thresholds[min(best_idx, len(thresholds)-1)]

roc_raw, pr_raw, f1_raw, thresh_raw = compute_metrics(y_true, y_score)
roc_pp, pr_pp, f1_pp, thresh_pp = compute_metrics(y_true, y_score_pp)

print("==================================================")
print("             RESULTADOS DO PIPELINE SOTA          ")
print("==================================================")
print(f"Modelo Sem Post-Processing:")
print(f"  ROC-AUC : {roc_raw:.4f}")
print(f"  PR-AUC  : {pr_raw:.4f}")
print(f"  F1-Score: {f1_raw:.4f} (Limiar = {thresh_raw:.3f})")
print(f"
Modelo COM Post-Processing por Cliente (FraudSquad):")
print(f"  ROC-AUC : {roc_pp:.4f}")
print(f"  PR-AUC  : {pr_pp:.4f}")
print(f"  F1-Score: {f1_pp:.4f} (Limiar = {thresh_pp:.3f})")
print("==================================================")


In [ ]:
# Visualização da Curva Precision-Recall e ROC
prec, rec, _ = precision_recall_curve(y_true, y_score_pp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Curva Precision-Recall
ax1.plot(rec, prec, color='#E63946', label=f'PR-AUC = {pr_pp:.4f}')
ax1.axhline(y_true.mean(), color='gray', linestyle='--', label=f'Baseline Aleatório ({y_true.mean():.3f})')
ax1.set_xlabel('Recall')
ax1.set_ylabel('Precision')
ax1.set_title('Curva Precision-Recall (Validação Temporal)')
ax1.legend()

# Curva ROC
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_true, y_score_pp)
ax2.plot(fpr, tpr, color='#1D3557', label=f'ROC-AUC = {roc_pp:.4f}')
ax2.plot([0, 1], [0, 1], color='gray', linestyle='--')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('Curva ROC (Validação Temporal)')
ax2.legend()

plt.tight_layout()
plt.show()
